# Hidden Number Predictor

## Goal
Given a fixed size array of numbers, predict a number n such that the sum of all numbers + n = some hidden target.

## Architecture
A simple **Single-Layer Neural Network (Perceptron)** will suffice for this simple problem.

Instead of telling AI the formula, we will give it random weights (guesses) and a bias. It will look at the numbers, make a prediction, see how wrong it was (the **Loss**), and adjust its guesses slightly using **Gradient Descent**

## Step 1: Import Libraries and Create Data
Here our hidden target is **100**.

First we create 1000 sample data points (shape (1000,3) tensor)
Each data point (row) has 3 numbers. The sum of these three and the prediction should equal the hidden target.

In [1]:
# Import required libraries and create data
import torch
import torch.nn as nn
import torch.optim as optim

# Set seed manually to make the code reproducible
torch.manual_seed(67)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# The hidden target the AI needs to discover
HIDDEN_TARGET = 100.0

# Generate random input data (1000 rows of 3 numbers each)
# X represents our inputs (x1, x2, x3)
X = torch.randn(1000,3) * 10

# Calculate the correct number the AI needs to output
# Sum across every row (dim=1) and keep the output in same dimension (1000, 1)
y = HIDDEN_TARGET - X.sum(dim=1, keepdim=True)

print("Sample input (X):\n", X[0])
print("Sample correct output (y):\n", y[0])


Sample input (X):
 tensor([-10.3901,  -4.2980,  15.2507])
Sample correct output (y):
 tensor([99.4374])


## Step 2: Create a simple linear model
The model takes 3 inputs and produces 1 output. The forward method simply calls `linear()` with the given inputs. 

In [2]:
# Create a simple linear model
class SimpleAI(nn.Module):
    def __init__(self):
        super().__init__()
        # 3 input numbers, 1 output
        self.linear = nn.Linear(in_features=3, out_features=1)

    def forward(self,x):
        return self.linear(x)

# Initialize our AI
model = SimpleAI().to(device)

## Step 3: Define the Loss and Optimizer
The `criterion` calculates the error (using MSE) while the `optimizer` actually updates the parameters.

Here, Adaptive Moment Estimation (**Adam**) is used. Adam has several practical benefits:
- **Dynamically adjusts each parameter**: Every single weight in a neural network gets its own custom, changing learning rate. If one weight needs to change quickly and another needs to change slowly, the algorithm automatically handles it.
- **First Moment** (mean): This acts like **momentum**. It tracks the average direction the gradient has been moving. If it keeps moving in the same direction, it speeds up. If not, then it slows down to stay steady.
- **Second Moment** (uncentered variance): It tracks how **violently** the gradients are moving. If a specific weight is experiencing massive, erratic jumps, the algorithm shrinks its step size to prevent the model from crashing
- **Exponentially decaying averages**: The algorithm cares much more about recent changes than steps taken a long time ago.

In [3]:
# Define the loss and optimizer

# Calculates loss using mean squared error (MSE)
criterion = nn.MSELoss()

# Updates the parameters using Adam (Adaptive Moment Estimation)
optimizer = optim.Adam(model.parameters(), lr=0.1) # lr = learning rate

## Step 4: The Training Loop
We'll loop over the data 2500 times (epochs)
1. Clear the gradients from previous epoch. PyTorch accumulates it by default.
2. **Forward Pass**: Run the data through the model and make a prediction. It happens all at once.
3. Calculate the error.
4. **Backward Pass**: Calculate the parameter adjustments based on the error using backpropagation
5. Update the parameters

In [4]:
# Move training data to the target device
X_train = X.to(device)
y_train = y.to(device)

EPOCH_RANGE = 2500
for epoch in range(EPOCH_RANGE):
    # Reset gradients
    optimizer.zero_grad()

    # Forward pass: make a prediction
    predictions = model(X_train) # shape: (1000,1)

    # Calculate the error (loss)
    loss = criterion(predictions, y_train)

    # Backward pass: Calculate how to adjust weights to reduce the error
    loss.backward()

    # Update the AI's internal parameters
    optimizer.step()

    # Print progress every 1/20th of the total epochs
    if (epoch + 1) % (EPOCH_RANGE//20) == 0:
        print(f"Epoch [{epoch+1}/{EPOCH_RANGE}], Loss: {loss.item():.4f}")

Epoch [125/2500], Loss: 7723.9697
Epoch [250/2500], Loss: 5847.9351
Epoch [375/2500], Loss: 4340.4380
Epoch [500/2500], Loss: 3151.2190
Epoch [625/2500], Loss: 2232.4954
Epoch [750/2500], Loss: 1539.3433
Epoch [875/2500], Loss: 1030.1494
Epoch [1000/2500], Loss: 667.1188
Epoch [1125/2500], Loss: 416.7836
Epoch [1250/2500], Loss: 250.4077
Epoch [1375/2500], Loss: 144.2169
Epoch [1500/2500], Loss: 79.3600
Epoch [1625/2500], Loss: 41.5892
Epoch [1750/2500], Loss: 20.6869
Epoch [1875/2500], Loss: 9.7337
Epoch [2000/2500], Loss: 4.3169
Epoch [2125/2500], Loss: 1.7980
Epoch [2250/2500], Loss: 0.7005
Epoch [2375/2500], Loss: 0.2542
Epoch [2500/2500], Loss: 0.0855


## Step 5: Test
Give the AI new numbers and it guesses the right number to make the total sum **100**.

First we need to switch the model to `eval()` evaluation mode.

In [5]:
# New random test tensor 
test_input = (torch.randn(1000, 3) * 10).to(device)
output = HIDDEN_TARGET - test_input.sum(1, keepdim=True)

model.eval() # Put model in evaluation mode
with torch.no_grad():
    prediction = model(test_input)

# Calculate the mean absolute error
print("Average error: ", torch.abs(output - prediction).mean().item())

Average error:  0.2914052903652191


## Learning the secret formula
In this simple example, the formulae for the output is simple enough: `output = 100 - x1 - x2 - x3` or `output = (-1)*x1 + (-1)*x2 + (-1)*x3 + 100`. This means the weights should `[-1,-1,-1]` and the bias be `100`. Let's verify this below. 

In [6]:
weights = model.linear.weight.data.squeeze().tolist()
bias = model.linear.bias.data.item()

print(f"Learned weights: {weights[0]:.2f}, {weights[1]:.2f}, {weights[2]:.2f}")
print(f"Learned bias: {bias:.2f}")

Learned weights: -1.00, -1.00, -1.00
Learned bias: 99.71
